# Solutions — Custom Hooks

One solution per exercise and per mini challenge, in lesson order.
Read these **after** you have tried. A solution you have not attempted teaches nothing.

Where an exercise happens in the playground, the solution is the code to put in
`playground/src/experiments/27-custom-hooks.jsx`, shown as a non-runnable block.

### LESSON 62 — Exercise

**1. Renaming `useToggle` to `getToggle`.** It still works. React does not inspect the name at
runtime; the Hook rules are enforced by the linter and by call order, not by a string.

What you lose is everything the convention buys:

- A reader scanning `const [on, t] = getToggle()` has no way to know that line holds state. The
  `use` prefix is the only signal that a component's state or Effects might hide inside a call.
- The ESLint rules match on the prefix. `getToggle` is, as far as the linter is concerned, a
  plain function — so calling it inside an `if` or a loop produces no warning at all, and you
  find out at runtime when the Hook order changes between renders.

That second point is the real cost: you have not just made the code less readable, you have
turned off the check that catches the worst Hook bug.

**2. Returning an object.**

```jsx
function useToggle(initial = false) {
  const [on, setOn] = useState(initial);
  const toggle = () => setOn((current) => !current);
  return { on, toggle };
}

function Panel({ title }) {
  const { on, toggle } = useToggle(false);
  // …
}

function useSearchBox(initial = "") {
  const [text, setText] = useState(initial);
  const query = useDebounced(text);
  const tips = useToggle(false);          // or: const { on: showTips, toggle: toggleTips }
  return { text, setText, query, showTips: tips.on, toggleTips: tips.toggle };
}
```

The array reads better **here**, and the reason is visible in `useSearchBox`: with two toggles in
one place you have to rename anyway, and the array lets you do it in one line
(`const [showTips, toggleTips] = useToggle()`) instead of a rename-on-destructure.

That is exactly why `useState` returns an array. Every component calls it more than once, and
`const [name, setName] = useState("")` beside `const [age, setAge] = useState(0)` only works
because the caller names the pair. An object shape would force
`const { value: name, setValue: setName } = useState("")` every time.

Use an object once there are more than two values, or when callers will usually want only some
of them — `const { query } = useSearchBox()` is fine and would be meaningless with an array.

**3. `useOnlineStatus`.**

```jsx
function useOnlineStatus() {
  const [online, setOnline] = useState(navigator.onLine);

  useEffect(() => {
    const goOnline = () => setOnline(true);
    const goOffline = () => setOnline(false);

    window.addEventListener("online", goOnline);
    window.addEventListener("offline", goOffline);

    return () => {
      window.removeEventListener("online", goOnline);
      window.removeEventListener("offline", goOffline);
    };
  }, []);

  return online;
}

// in the component:
const online = useOnlineStatus();
return <p>You are {online ? "online" : "offline"}.</p>;
```

Three things worth noticing. The listener functions are declared **inside** the Effect so that
the cleanup removes the same function references it added — `removeEventListener` compares by
identity, so re-creating the arrow inline in both calls would remove nothing. The deps array is
empty because nothing from the render scope is used. And the Hook returns a single boolean, not
a pair: there is nothing for the caller to set, which is a good sign the single-value shape is
right.

To test it, use Chrome DevTools → Network → the throttling dropdown → **Offline**.

**Common mistake:** reading `navigator.onLine` in the Effect body and setting state from it,
instead of using it as the initial value. That gives you one pointless extra render and still
does not tell you about later changes — the events do that.

**4. What is wrong with `useMount` and `useEffectOnce`.**

They are named after React's lifecycle, not after a concrete high-level use case. `useMount(fn)`
says only *when* the function runs; it could be doing anything — logging, subscribing, fetching,
starting a timer. It constrains nothing, so the calling code is no more declarative than the raw
`useEffect` it wraps, and you have added a layer that must be learned without removing anything
that had to be understood.

A Hook worth having says what job it does, which is why React's counter-examples are `useData`,
`useImpressionLog` and `useChatRoom` — each of those can only do one thing.

For "log a page view when this screen appears": `usePageViewLog(pageName)`, or
`useImpressionLog("checkout")` in React's own vocabulary. The name now states the job, and the
fact that it happens on mount is an implementation detail inside the Hook, where it belongs.

### LESSON 62 — Mini challenge

1. **Websocket subscribe + cleanup in three components** — extract. There is an Effect, it is
   genuinely repeated, and it has a name: `useChatRoom(roomId)` or `useLiveUpdates(channel)`.
   This is the clearest case there is.

2. **`useState(false)` in two components** — do **not** extract. This is the "some duplication
   is fine" case in its purest form: one line, no Effect, nothing to name. A `useToggle` is not
   *wrong*, but writing it to remove two identical lines is not the reason to have it.

3. **Five components fetching different URLs with the same loading/error/empty handling** —
   extract. `useData(url)` or `useFetch(url)`, returning `{ data, loading, error }`. The URL
   varies, which is exactly what a parameter is for; the plumbing that repeats is the part you
   are removing.

4. **One component, forty lines of fetch plumbing above eight lines of JSX** — extract, even
   though it is used once. `useProjectList()` and a component that is now nine lines. React's
   own guidance is that whenever you write an Effect you should consider wrapping it in a
   custom Hook, and repetition is not a precondition.

5. **Two components formatting a date the same way** — extract a **plain function**, not a Hook.
   `formatDate(value)` calls no Hooks, so naming it `useFormatDate` would be actively harmful:
   it would announce state that does not exist and subject a pure function to the Rules of Hooks.

**What distinguishes 2 from 4.** Not how many times the code appears — that is the axis the
question is designed to break. It is **whether there is anything to name**. Number 2 is one line
whose meaning is already obvious; a Hook adds a layer and hides nothing worth hiding. Number 4
is forty lines of *how* sitting on top of the component's *what*; extracting it gives that mess a
name and leaves a component you can read in one glance. Extract to remove noise or to name a
job, not to satisfy a duplicate count.

### LESSON 63 — Exercise

Parts 1 and 2 are runnable; part 3 is the playground.

In [ ]:
// L63 solution — safe read and write

function l63ReadJSON(key, fallback) {
  const stored = localStorage.getItem(key);
  if (stored === null) return fallback;        // missing key — NOT the same as bad data
  try {
    return JSON.parse(stored);
  } catch {
    return fallback;                           // corrupt data — do not crash the app
  }
}

function l63WriteJSON(key, value) {
  localStorage.setItem(key, JSON.stringify(value));
}

// --- 1. three ways ---------------------------------------------------------
l63WriteJSON("l63-good", { theme: "dark" });
localStorage.setItem("l63-bad", "{oops");

console.log("good value :", l63ReadJSON("l63-good", "FALLBACK"));
console.log("missing key:", l63ReadJSON("l63-absent", "FALLBACK"));
console.log("bad JSON   :", l63ReadJSON("l63-bad", "FALLBACK"));

// --- 2. round-trip an array of objects --------------------------------------
const l63People = [
  { id: 1, name: "Ada", active: true },
  { id: 2, name: "Grace", active: false },
  { id: 3, name: "Alan", active: true },
];

l63WriteJSON("l63-people", l63People);
const l63Back = l63ReadJSON("l63-people", []);

console.log("round-tripped:", l63Back);
console.log("deep equal:", JSON.stringify(l63Back) === JSON.stringify(l63People));

// clean up so re-running the notebook starts fresh
["l63-good", "l63-bad", "l63-people"].forEach((k) => localStorage.removeItem(k));

**Common mistake:** writing `if (!stored) return fallback;`. Through `JSON.stringify` it happens
to survive — `0` serialises to `"0"` and `""` to `'""'`, both truthy strings — so the bug sits
there silently until the day you store a raw string and a saved empty value starts reading as
"nothing saved". Only `null` means "not there", so test for `null`.

A second one: putting the whole thing in `try`/`catch` and returning the fallback for any
failure. That works, but it hides the difference between "nothing saved yet" and "the saved data
is corrupt" — and those deserve different handling once the app grows.

**3. `useLocalStorage` in the playground.**

```jsx
function useLocalStorage(key, initialValue) {
  const [value, setValue] = useState(() => {
    const stored = localStorage.getItem(key);
    if (stored === null) return initialValue;
    try {
      return JSON.parse(stored);
    } catch {
      return initialValue;
    }
  });

  useEffect(() => {
    localStorage.setItem(key, JSON.stringify(value));
  }, [key, value]);

  return [value, setValue];
}

// then, inside useSearchBox:
const [showTips, setShowTips] = useLocalStorage("tips-open", false);
const toggleTips = () => setShowTips((current) => !current);
```

Reload the page with the tips open and they stay open. Note that `useToggle` is no longer doing
that job — `useLocalStorage` returns a setter, not a toggle, so the toggling moves up one level.
An alternative worth considering is a `usePersistentToggle` that composes the two, which is
LESSON 64's subject.

### LESSON 63 — Mini challenge

In [ ]:
// L63 solution — eager vs lazy initial value

function l63ReadStorage() {
  console.log("  reading storage");
  return JSON.parse('{"count":7}');
}

function l63Eager(value) {
  return value;                 // the argument was already evaluated by the caller
}

function l63Lazy(init) {
  return typeof init === "function" ? init() : init;
}

console.log("three eager calls:");
const l63E1 = l63Eager(l63ReadStorage());
const l63E2 = l63Eager(l63ReadStorage());
const l63E3 = l63Eager(l63ReadStorage());

console.log("three lazy calls (as React would, keeping the FIRST result):");
let l63Slot = undefined;
function l63LazyOnce(init) {
  if (l63Slot === undefined) l63Slot = l63Lazy(init);   // stands in for React's state slot
  return l63Slot;
}
const l63L1 = l63LazyOnce(l63ReadStorage);
const l63L2 = l63LazyOnce(l63ReadStorage);
const l63L3 = l63LazyOnce(l63ReadStorage);

console.log("eager values:", l63E1.count, l63E2.count, l63E3.count);
console.log("lazy values :", l63L1.count, l63L2.count, l63L3.count);

// Answer:
// Three "reading storage" lines for eager, one for lazy.
//
// The state VALUE is identical either way, because React ignores the initial argument on every
// render after the first. The difference is who does the work. useState(readStorage()) is a
// call expression: JavaScript evaluates it BEFORE useState is entered, so storage is read and
// parsed on every single render and the result is thrown away every time but the first.
// useState(readStorage) passes the function itself, and React only calls it on the initial
// render. Same state, but one of them pays for a read and a JSON.parse on every keystroke.

**Common mistake:** writing `useState(() => readStorage())` and thinking the arrow is what makes
it lazy in some special React way. The arrow is lazy for the ordinary JavaScript reason — the
body has not run yet. `useState(readStorage)` is the same thing with one fewer wrapper. The
arrow is only needed when you have arguments to pass: `useState(() => readStorage(key))`.

### LESSON 64 — Exercise

In [ ]:
// L64 solution — two instances vs one shared instance

function l64MakeCounter(start = 0) {
  let count = start;
  return {
    read: () => count,
    increment: () => { count += 1; },
    reset: () => { count = start; },
  };
}

// --- two separate instances -------------------------------------------------
const l64First = l64MakeCounter();
const l64Second = l64MakeCounter();

l64First.increment();
l64First.increment();
l64First.increment();
l64First.increment();
l64Second.increment();

console.log("two instances — first:", l64First.read(), "second:", l64Second.read());

// --- one instance, handed to two consumers ----------------------------------
const l64Shared = l64MakeCounter();

function l64ConsumerA(counter) { counter.increment(); }
function l64ConsumerB(counter) { counter.increment(); }

l64ConsumerA(l64Shared);
l64ConsumerB(l64Shared);

console.log("one instance, two consumers:", l64Shared.read());

l64First.reset();
console.log("after reset — first:", l64First.read(), "second:", l64Second.read());

// Which arrangement matches a custom Hook?
//   The FIRST one. Every component that calls useCounter() runs the factory again and gets its
//   own box. The second arrangement — one instance passed to two consumers — is what lifting
//   state up (L29) or Context (L56) does, and it is the only way to get shared state.

**Common mistake:** expecting `reset()` on one counter to affect the other, or reading
`l64First.read()` once into a variable and expecting that variable to change later. The object
is shared, but a value read out of it is a snapshot — the same distinction as state versus the
value you destructured from it during a render.

**2 and 3 — in the playground.**

```jsx
// 2. a third panel: independent, like the other two
<Panel title="first" />
<Panel title="second" />
<Panel title="third" />

// 3. two panels sharing ONE toggle, the third still independent
function SharedPanel({ title, open, toggle }) {
  return (
    <p>
      <button onClick={toggle}>{open ? "hide" : "show"} {title}</button>{" "}
      {open && <span>contents of {title}</span>}
    </p>
  );
}

export default function Experiment27() {
  const [bothOpen, toggleBoth] = useToggle(false);   // called ONCE, up here
  const search = useSearchBox("");

  return (
    <div>
      <SharedPanel title="first" open={bothOpen} toggle={toggleBoth} />
      <SharedPanel title="second" open={bothOpen} toggle={toggleBoth} />
      <Panel title="third" />                        {/* its own useToggle */}
      {/* … */}
    </div>
  );
}
```

`useToggle` is unchanged. What changed is **where it is called**: once, in the closest common
parent, with the value passed down. That is LESSON 29's "lifting state up", and the fact that
the state happens to live inside a custom Hook makes no difference to it. The third panel calls
`useToggle` itself and opens on its own — both arrangements, same Hook, side by side.

### LESSON 64 — Mini challenge

In [ ]:
// L64 solution — the "one Hook for everything" mistake

function l64MakeAppState() {
  let user = { name: "Ada" };
  let theme = "light";
  let cart = [];
  let query = "";

  return {
    read: () => ({ user, theme, cart, query }),
    setTheme: (next) => { theme = next; },
    addToCart: (item) => { cart = [...cart, item]; },
  };
}

const l64AppA = l64MakeAppState();
const l64AppB = l64MakeAppState();

l64AppA.setTheme("dark");
l64AppA.addToCart("book");

console.log("instance A:", l64AppA.read());
console.log("instance B:", l64AppB.read());
console.log("same cart?", l64AppA.read().cart.length === l64AppB.read().cart.length);

// 1. A component that only needs the theme also gets the user, the cart and the search query.
//    In React that means it subscribes to all of them: every cart change re-renders a component
//    that only ever displays a colour. The Hook makes it impossible to depend on less.
//
// 2. Calling it in two components fails to share the cart because each call runs the whole Hook
//    again and creates a separate set of state. Custom Hooks share stateful LOGIC, not state.
//    Two calls are two independent boxes, exactly as the output above shows.
//
// 3. "concrete high-level" — React's phrase is to keep custom Hooks focused on concrete
//    high-level use cases. The fix is to split it into useCurrentUser(), useTheme(), useCart()
//    and useSearchQuery(), each with one job — and for the parts that genuinely must be shared
//    across the app (the user, the cart), to put the state in a Context provider and let the
//    Hook read it, so there is one owner and many readers.